In [0]:
from pyspark.sql.types import IntegerType, DoubleType, FloatType
from pyspark.sql.functions import col, count, when, isnan, to_timestamp, date_format, from_utc_timestamp
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("year", "2025", "Year (YYYY)")
dbutils.widgets.text("month", "01", "Month (MM)")

year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")

In [0]:
# year = "2025"
# month = "01"
batch_processing = f"{year}{str(month).zfill(2)}"

batch_processing

In [0]:
source_table = "oftalmo_sus.01_bronze.tb_datasus_sia_pa"
sia_pa_data = spark.table(source_table).select(
    "PA_CMP",
    "PA_MUNPCN",
    "PA_PROC_ID",
    "PA_QTDPRO",
    "PA_VALPRO",
    "PA_IDADE",
    "PA_SEXO",
    "batch_processing"
)
display(sia_pa_data.limit(10))

In [0]:
oftalmo_data = sia_pa_data.filter(
    sia_pa_data.PA_PROC_ID.startswith("0405") |
    sia_pa_data.PA_PROC_ID.startswith("030305") |
    sia_pa_data.PA_PROC_ID.startswith("021106")
)
display(oftalmo_data.limit(10))

In [0]:
oftalmo_data_typed = (
    oftalmo_data
    .withColumn("PA_QTDPRO", col("PA_QTDPRO").cast(IntegerType()))
    .withColumn("PA_IDADE", col("PA_IDADE").cast(IntegerType()))
    .withColumn("PA_VALPRO", col("PA_VALPRO").cast(DoubleType()))
)

display(oftalmo_data_typed.limit(10))

In [0]:
oftalmo_data_typed.select([
    count(when(
        col(c).isNull() | (isnan(c) if isinstance(oftalmo_data_typed.schema[c].dataType, (IntegerType, DoubleType, FloatType)) else False),c)).alias(c) 
    for c in oftalmo_data_typed.columns
]).display()

In [0]:
required_columns = [
    "PA_CMP",     # Competência (anomes)
    "PA_PROC_ID",  # Código do Procedimento
    "PA_MUNPCN",   # Código do Município
    "PA_VALPRO"    # Valor do Procedimento
]

oftalmo_data_clened = oftalmo_data_typed.na.drop(subset=required_columns)
deleted_rows = oftalmo_data_typed.count() - oftalmo_data_clened.count()

if deleted_rows > 0:
    print(f"⚠️ Atenção: {deleted_rows} registros nulos foram descartados para garantir a qualidade.")
else:
    print("✅ Qualidade confirmada: Nenhum registro nulo encontrado neste lote.")

In [0]:
target_table = "oftalmo_sus.02_silver.tb_fato_sia_oftalmo"
replace_condition = f"batch_processing = '{batch_processing}'"

(
    oftalmo_data_clened.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("PA_CMP")
    .option("replaceWhere", replace_condition) # Protegendo o resto do Data Lake
    .saveAsTable(target_table)
)

print("✅ Camada Silver concluída! Tabela Fato salva com sucesso.")

In [0]:
delta_table = DeltaTable.forName(spark, target_table)

history_df = delta_table.history()

display(
    history_df
    .orderBy(col("version").desc())
    .limit(1)
    .select(
        col("version"),
        date_format(from_utc_timestamp(col("timestamp"), "America/Sao_Paulo"), "dd-MM-yyyy HH:mm:ss").alias("timestamp"),
        col("operation"),
        col("operationMetrics.insert").alias("insert"),
        col("operationMetrics.delete").alias("delete"),
        col("operationMetrics.update").alias("update"),
        col("operationMetrics.numOutputRows").alias("num_rows")
    )
)